# `contextual-turn-encoder-base` — generación de embeddings base (f1) sobre Dialog2Flow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jumafernandez/doctorado-unsl/blob/main/packages/contextual-turn-embeddings/training/contextual-turn-encoder-base/01_generate_base_embeddings_colab.ipynb)

Genera los **embeddings base `e_t`** del corpus **completo** de Dialog2Flow
(`sergioburdisso/dialog2flow-dataset`, ~3,4M turnos) con el encoder
`sergioburdisso/dialog2flow-joint-bert-base` (768-d) — el **mismo** que usan las baselines ANN
(apples-to-apples).

**Diseño:** guarda directamente un `.npy` memmapeado y hace **checkpoint cada ~500k turnos**
cortando en **borde de diálogo** (como el corte del 1M). Es **resumible**: si Colab se desconecta,
volvé a correr y continúa desde el último checkpoint.

> Pesado → correr en **Colab con GPU**. El entrenamiento (f2) va en el M2 (notebook 02).
> No descarga modelos de generación de texto; solo el encoder D2F + el dataset.

## 1. Instalación

In [ ]:
!pip install -q "datasets==2.21.0" sentence-transformers

## 2. Configuración

In [ ]:
import os

REPO_ID = "sergioburdisso/dialog2flow-dataset"
ENCODER = "sergioburdisso/dialog2flow-joint-bert-base"   # base encoder f1 (768-d)

# Orden EXACTO de configs (igual que notebook_01 de ANN-UNSL).
DATASETS = ["ABCD", "BiTOD", "Disambiguation", "DSTC2-Clean", "FRAMES", "GECOR",
            "HDSA-Dialog", "KETOD", "MS-DC", "MulDoGO", "MultiWOZ_2.1", "MULTIWOZ2_2",
            "SGD", "SimJointGEN", "SimJointMovie", "SimJointRestaurant",
            "Taskmaster1", "Taskmaster2", "Taskmaster3", "WOZ2_0"]

# >>> ajustá esta ruta de salida (Drive) <<<
BASE_DIR = "/content/drive/MyDrive/Doctorado/contextual-turn-encoder-base/d2f-full"
CHECKPOINT_EVERY = 500_000     # turnos por checkpoint (corta en borde de diálogo)
ENC_BATCH = 64
os.makedirs(BASE_DIR, exist_ok=True)
print("salida:", BASE_DIR)

## 3. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(BASE_DIR, exist_ok=True)

## 4. Descargar el corpus completo y armar la metadata canónica

Recorre los 20 configs (**sin** el corte de 1M) y arma el DataFrame canónico, con el mismo
`dialogue_id = f"{config}_{split}_{i}"` que el benchmark. Resumible: si ya existe, lo carga.

In [ ]:
import pandas as pd
from datasets import load_dataset

META_PATH = os.path.join(BASE_DIR, "dialogs-full.pkl")

if os.path.exists(META_PATH):
    df = pd.read_pickle(META_PATH)
    print("metadata ya existe:", df.shape)
else:
    rows = []
    for config in DATASETS:
        print("procesando:", config, flush=True)
        ds = load_dataset(REPO_ID, config, trust_remote_code=True)
        for split in ds.keys():
            for i, item in enumerate(ds[split]):
                dialogue_id = f"{config}_{split}_{i}"
                for turn_id, turn in enumerate(item["dialog"]):
                    rows.append({"dataset": config, "split": split,
                                 "dialogue_id": dialogue_id, "turn_id": turn_id,
                                 "speaker": turn.get("speaker"),
                                 "utterance": turn.get("text", "")})
        print("   turnos acumulados:", len(rows), flush=True)
    df = pd.DataFrame(rows)
    df.to_pickle(META_PATH)
    print("metadata guardada:", df.shape)

import numpy as np
np.save(os.path.join(BASE_DIR, "ids.npy"), np.arange(len(df), dtype="int64"))
print("turnos:", len(df), "| diálogos:", df["dialogue_id"].nunique())

## 5. Generar embeddings base con checkpointing resumible

Escribe un `.npy` memmapeado (`base_embeddings.npy`) y guarda un `manifest.json` con cuántos
turnos van hechos. Cada chunk corta en **borde de diálogo** y tiene ≤ `CHECKPOINT_EVERY` turnos.

In [ ]:
import json
import numpy as np
import torch
from numpy.lib.format import open_memmap
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
model = SentenceTransformer(ENCODER, device=device)
D = model.get_sentence_embedding_dimension()
N = len(df)
print("encoder dim:", D, "| turnos:", N)

# Bordes de diálogo (fin exclusivo de cada diálogo) en el orden del corpus.
dids = df["dialogue_id"].to_numpy()
ends = np.concatenate([np.where(dids[1:] != dids[:-1])[0] + 1, [N]]).astype(np.int64)


def next_end(done, target):
    cand = ends[(ends > done) & (ends <= done + target)]
    if len(cand):
        return int(cand[-1])
    return int(ends[ends > done][0])   # diálogo más largo que target: hasta su borde


EMB_PATH = os.path.join(BASE_DIR, "base_embeddings.npy")
MAN_PATH = os.path.join(BASE_DIR, "manifest.json")

if os.path.exists(EMB_PATH) and os.path.exists(MAN_PATH):
    man = json.load(open(MAN_PATH))
    assert man["n_total"] == N and man["model_dim"] == D, "manifest no coincide con los datos"
    done = int(man["done"])
    emb = open_memmap(EMB_PATH, mode="r+")
    print(f"reanudando desde turno {done}/{N}")
else:
    emb = open_memmap(EMB_PATH, mode="w+", dtype="float32", shape=(N, D))
    done = 0
    json.dump({"done": 0, "n_total": N, "model_dim": D, "encoder": ENCODER}, open(MAN_PATH, "w"))

utterances = df["utterance"].fillna("").astype(str).tolist()
while done < N:
    end = next_end(done, CHECKPOINT_EVERY)
    vecs = model.encode(utterances[done:end], batch_size=ENC_BATCH, convert_to_numpy=True,
                        show_progress_bar=True, device=device).astype("float32")
    emb[done:end] = vecs
    emb.flush()
    done = end
    json.dump({"done": done, "n_total": N, "model_dim": D, "encoder": ENCODER}, open(MAN_PATH, "w"))
    print(f"checkpoint: {done}/{N} turnos ({100 * done / N:.1f}%)", flush=True)
print("Generación COMPLETA.")

## 6. Verificación

In [ ]:
emb = np.load(os.path.join(BASE_DIR, "base_embeddings.npy"), mmap_mode="r")
print("base_embeddings:", emb.shape, emb.dtype)
sample = np.asarray(emb[np.random.default_rng(0).choice(len(emb), size=10000, replace=False)])
print("NaNs (muestra):", bool(np.isnan(sample).any()))
print("filas == metadata:", emb.shape[0] == len(df))
print("Listo. Subí base_embeddings.npy + dialogs-full.pkl + ids.npy a donde uses el M2.")